# Random forest
## Baseline

In [ ]:
from pathlib import Path
import json
import pandas as pd
import os

DATA_DIR = Path("lasftm_asia")
FEATURES_JSON = DATA_DIR / "lastfm_asia_features.json"
TARGET_CSV = DATA_DIR / "lastfm_asia_target.csv"
OUT_ONEHOT_CSV = DATA_DIR / "lastfm_asia_features_onehot.csv"


if os.path.exists(OUT_ONEHOT_CSV):
    onehot_df = pd.read_csv(OUT_ONEHOT_CSV)

else:


    with FEATURES_JSON.open("r") as f:
        raw_features = json.load(f)

    rows = [
        (int(user_id), int(artist_id), 1)
        for user_id, artist_ids in raw_features.items()
        for artist_id in artist_ids
    ]
    long_df = pd.DataFrame(rows, columns=["user_id", "artist_id", "value"])

    # Pivot to one-hot format:
    onehot_df = (
        long_df.pivot_table(
            index="user_id",
            columns="artist_id",
            values="value",
            aggfunc="max",
            fill_value=0,
        )
        .astype("int8")
        .sort_index()
    )

    # Rename columns
    onehot_df.columns = [f"artist_{int(c)}" for c in onehot_df.columns]
    onehot_df = onehot_df.reset_index()

    target_df = pd.read_csv(TARGET_CSV)
    onehot_df = onehot_df.merge(target_df, how="inner", left_on="user_id", right_on="id").drop(columns=["id"])

    onehot_df.to_csv(OUT_ONEHOT_CSV, index=False)

print(f"Saved: {OUT_ONEHOT_CSV}")
print(f"Shape: {onehot_df.shape}")
onehot_df.head()


Saved: lasftm_asia/lastfm_asia_features_onehot.csv
Shape: (7451, 7844)


,user_id,artist_0,artist_1,artist_2,artist_3,artist_4,artist_5,artist_6,artist_7,artist_8,...,artist_7833,artist_7834,artist_7835,artist_7836,artist_7837,artist_7838,artist_7839,artist_7840,artist_7841,target
0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,8
1,1,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,17
2,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
3,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,17
4,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score


X = onehot_df.drop(columns=["target"]).to_numpy()
y = onehot_df["target"].to_numpy()

rf_model = RandomForestClassifier(n_jobs=-1)
xg_model = GradientBoostingClassifier()

rf_scores = cross_val_score(rf_model, X, y, cv=15, scoring="accuracy")
xg_scores = cross_val_score(xg_model, X, y, cv=15, scoring="accuracy")



TypeError: GradientBoostingClassifier.__init__() got an unexpected keyword argument 'n_jobs'

In [ ]:
print(f"Random Forest: {rf_scores.mean():.4f} +- {rf_scores.std():.4f}")
print(f"XGBoost: {xg_scores.mean():.4f} +- {xg_scores.std():.4f}")



0.7332 +- 0.0159
